# 07 · Knowledge distillation and catastrophic forgetting

Two things every fine-tuning project runs into. **Distillation**: make a small, cheap model imitate a big one —
here with soft targets at a temperature, compared against hard labels. **Catastrophic forgetting**: train on a new
task and watch the old one collapse — then compare the standard mitigations on one chart.

**Time:** ~25 min · **Runs:** offline on CPU in < 1 min, no downloads · **Needs:**
[03 · LoRA from scratch](03_lora_from_scratch.ipynb) (the digits → rotated-digits setup),
[Training loops and optimizers](../06_deep_learning/02_training_loop_optimizers_lr_schedules.ipynb)

## Why this matters in interviews

- "What is knowledge distillation?" comes with two traps: is it a quality or a cost project (cost), and can the
  student beat the teacher (not on the distilled distribution).
- "Why is the T² there?" and "what are soft targets telling the student?" are the technical follow-ups.
- "Our fine-tuned model got worse at everything else — why, and what do you do?" is catastrophic forgetting. The
  answer interviewers want is a ranked list of mitigations and a regression eval, not just "use LoRA".

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `tn10` | What is knowledge distillation? |
| `tn12` | What is catastrophic forgetting and how do you avoid it? |
| `tn30` | What is synthetic data, and when is it safe to use? |
| `tn39` | How do you decide between fine-tuning a small model and prompting a large one? |
| `tn20` | How do you evaluate a fine-tuned model before shipping it? (regression suite) |
| `tn21` | What is continued (domain-adaptive) pretraining? (mix in general data) |
| `tn27` | What is model merging, and does it actually work? |
| `sf29` | Design a system so that a competitor who steals your fine-tuned weights gets nothing useful (distillation via your API) |

In [ ]:
import copy, math, time
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

torch.set_num_threads(1)
torch.manual_seed(0); np.random.seed(0)

X, y = load_digits(return_X_y=True)
X = (X / 16.0).astype(np.float32)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
T_ = lambda a: torch.tensor(a)
print(f"digits: {len(Xtr)} train / {len(Xte)} test images of 8x8 pixels, 10 classes")

def mlp(hidden, depth=2):
    layers, d = [], 64
    for _ in range(depth):
        layers += [nn.Linear(d, hidden), nn.ReLU()]; d = hidden
    return nn.Sequential(*layers, nn.Linear(d, 10))

def accuracy(model, X_, y_):
    with torch.no_grad():
        return (model(T_(X_)).argmax(1).numpy() == y_).mean()

n_params = lambda m: sum(p.numel() for p in m.parameters())

## Part 1 — Knowledge distillation

**In plain English:** a big "teacher" model writes the answers and a small "student" learns from them. The student
ends up much smaller and nearly as good *on that task* — and inherits the teacher's mistakes.

Three flavours, most common first (`tn10`):

| Flavour | Student learns from | Needs | Typical use |
|---|---|---|---|
| **Response distillation** (synthetic data) | the teacher's generated text, via ordinary SFT | only the teacher's outputs (an API is enough) | by far the most common with LLMs: "have GPT-4-class write 20k answers, SFT a 7B on them" |
| **Logit distillation** | the teacher's full probability distribution per token (KL divergence) | the teacher's logits — open weights | richer signal per example; classic Hinton-style KD (below) |
| **Feature distillation** | the teacher's hidden states / attention maps | white-box access to both | compressing a specific architecture (e.g. DistilBERT) |

### The teacher

In [ ]:
torch.manual_seed(0)
teacher = mlp(256)
opt = torch.optim.Adam(teacher.parameters(), lr=1e-3)
Xt, yt = T_(Xtr), T_(ytr)
for epoch in range(40):
    perm = torch.randperm(len(Xt))
    for i in range(0, len(Xt), 64):
        idx = perm[i:i + 64]
        loss = F.cross_entropy(teacher(Xt[idx]), yt[idx])
        opt.zero_grad(); loss.backward(); opt.step()
teacher.eval()
print(f"teacher: {n_params(teacher):,} params, test accuracy {accuracy(teacher, Xte, yte):.3f}")

### Soft targets and temperature

The teacher's probabilities carry more than the label: *which wrong answers are nearly right*. A "3" that looks a bit
like an "8" gets some probability on 8. Dividing the logits by a **temperature** `T > 1` before the softmax spreads
the probability out, so that "dark knowledge" becomes visible to the student.

In [ ]:
with torch.no_grad():
    logits_te = teacher(T_(Xte))
probs = F.softmax(logits_te, -1)
k = int(torch.argmin(probs.max(1).values))             # the test image the teacher is least sure about
fig, axes = plt.subplots(1, 5, figsize=(11, 2.8), gridspec_kw={"width_ratios": [1, 2, 2, 2, 2]})
axes[0].imshow(Xte[k].reshape(8, 8), cmap="gray_r"); axes[0].set_title(f"true label {yte[k]}"); axes[0].axis("off")
for ax, temp in zip(axes[1:], [1, 2, 4, 8]):
    p = F.softmax(logits_te[k] / temp, -1).numpy()
    ax.bar(range(10), p, color=["#2f9e44" if c == yte[k] else "#adb5bd" for c in range(10)])
    ax.set_title(f"T = {temp}"); ax.set_xticks(range(10)); ax.set_ylim(0, 1)
axes[1].set_ylabel("teacher probability")
fig.suptitle("Soft targets: higher temperature reveals which classes the teacher finds similar"); plt.tight_layout(); plt.show()
entropy = lambda p: -(p * p.clamp_min(1e-12).log()).sum().item()
print("entropy of the teacher's distribution (nats):", {t: round(entropy(F.softmax(logits_te[k] / t, -1)), 2) for t in (1, 2, 4, 8)})

### The distillation loss

```
L = α · T² · KL( softmax(teacher/T) ‖ softmax(student/T) )  +  (1 − α) · CE(student, hard label)
```

Why **T²**: the gradient of the softened KL term scales like 1/T², so multiplying by T² keeps its size comparable to
the hard-label term when you change T (Hinton et al., 2015). Let us check that claim numerically:

In [ ]:
torch.manual_seed(0)
s_logits = torch.randn(256, 10, requires_grad=True)
t_logits = torch.randn(256, 10) * 3
def kd_term(student_logits, teacher_logits, T):
    return F.kl_div(F.log_softmax(student_logits / T, -1), F.softmax(teacher_logits / T, -1), reduction="batchmean")
for T in (1, 2, 4, 8):
    g_raw = torch.autograd.grad(kd_term(s_logits, t_logits, T), s_logits)[0].norm().item()
    g_scaled = torch.autograd.grad(T * T * kd_term(s_logits, t_logits, T), s_logits)[0].norm().item()
    print(f"T={T}: gradient norm without T² {g_raw:.4f}   with T² {g_scaled:.4f}")

Without the factor the soft-target gradient shrinks roughly like 1/T² (a little faster at low T); with it, it
stays of the same order across temperatures.

### Student: hard labels vs soft targets

The realistic distillation setting: labelled data is **scarce** (100 labelled images), but there is plenty of
**unlabelled** input (the other ~1,150 training images, labels hidden) that the teacher can label. The student is
~30× smaller than the teacher. Three ways to train it:

1. **hard labels only** — cross-entropy on the 100 labelled images;
2. **soft targets** — the distillation loss on the same 100 images;
3. **soft targets + transfer set** — the teacher also labels the unlabelled images (the logit-level version of
   "generate synthetic training data with the big model").

In [ ]:
rng = np.random.default_rng(0)
lab_idx = rng.choice(len(Xtr), size=100, replace=False)
unl_idx = np.setdiff1d(np.arange(len(Xtr)), lab_idx)
with torch.no_grad():
    teacher_logits_all = teacher(Xt)

def train_student(idx, use_soft, alpha=0.7, T=4.0, epochs=150, seed=0, hard_idx=None):
    """idx: images the student sees. Hard-label CE only on hard_idx (images whose labels we have)."""
    torch.manual_seed(seed)
    student = mlp(32, depth=1)
    opt_ = torch.optim.Adam(student.parameters(), lr=3e-3)
    Xs, ts = Xt[idx], teacher_logits_all[idx]
    has_label = torch.tensor(np.isin(idx, hard_idx if hard_idx is not None else idx))
    ys = yt[idx]
    for _ in range(epochs):
        out = student(Xs)
        hard = F.cross_entropy(out[has_label], ys[has_label])
        if use_soft:
            soft = T * T * F.kl_div(F.log_softmax(out / T, -1), F.softmax(ts / T, -1), reduction="batchmean")
            loss = alpha * soft + (1 - alpha) * hard
        else:
            loss = hard
        opt_.zero_grad(); loss.backward(); opt_.step()
    return student.eval()

t0 = time.time()
students = {}
for seed in range(3):                                 # 3 seeds: small-data results are noisy
    students.setdefault("hard labels\n(100 labelled)", []).append(train_student(lab_idx, use_soft=False, seed=seed))
    students.setdefault("soft targets\n(same 100)", []).append(train_student(lab_idx, use_soft=True, seed=seed))
    students.setdefault("soft targets +\nteacher-labelled\ntransfer set", []).append(
        train_student(np.concatenate([lab_idx, unl_idx]), use_soft=True, seed=seed, hard_idx=lab_idx))
print(f"trained 9 students in {time.time() - t0:.1f}s; student size {n_params(mlp(32, depth=1)):,} params vs teacher {n_params(teacher):,}")
kd_acc = {name: [accuracy(s, Xte, yte) for s in ss] for name, ss in students.items()}
for name, accs in kd_acc.items():
    print(f"{name.replace(chr(10), ' '):48s} accuracy {np.mean(accs):.3f} ± {np.std(accs):.3f}")

In [ ]:
names = list(kd_acc)
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.bar(names, [np.mean(kd_acc[n]) for n in names], yerr=[np.std(kd_acc[n]) for n in names], capsize=4,
       color=["#adb5bd", "#74c0fc", "#2f9e44"])
ax.axhline(accuracy(teacher, Xte, yte), color="#e8590c", ls="--", label=f"teacher ({n_params(teacher):,} params)")
ax.set_ylim(0.6, 1.0); ax.set_ylabel("student test accuracy (3 seeds)"); ax.legend(loc="lower right")
ax.set_title(f"Distilling into a {n_params(mlp(32, depth=1)):,}-parameter student"); plt.show()
assert np.mean(kd_acc[names[2]]) > np.mean(kd_acc[names[0]])
assert np.mean(kd_acc[names[2]]) <= accuracy(teacher, Xte, yte) + 0.01

**Read the chart.** The big win comes from the **transfer set**: the teacher turns unlabelled inputs into training
signal, and the student gets close to the teacher at a fraction of its size. Soft targets on the same 100 images
help less (compare the error bars before claiming a gain on data this small). And the student does not beat its
teacher: it can only be as good as the signal it copies.

### LLM-style distillation (what teams actually do)

```
prompts from your real traffic ──► big model (teacher) writes answers ──► filter / verify ──► SFT a small model
                                        (optionally several samples, keep the best by a judge or a test)
```

- It is **response distillation**: the student is fine-tuned (often with LoRA) on the teacher's text; no logits needed,
  so a hosted API can be the teacher.
- It is a **cost project**: use it when the pipeline already works with the big model and must get cheaper or faster.
  A 7B student distilled on a narrow task can approach a frontier model on *that task* at a fraction of the cost.
- Safeguards (`tn30`): verify or filter generated data; keep real human-written data in the mix; measure diversity;
  never evaluate on data generated by the same model family; **check the teacher's terms of service** — many
  providers forbid training competing models on outputs. The same mechanism is how a competitor can "steal" a model
  through its API (`sf29`), which is why rate limits and no raw logprobs matter.

## Part 2 — Catastrophic forgetting

**In plain English:** train a network intensively on something new and it quietly gets worse at what it knew — the
new gradients overwrite weights the old skill depended on.

**Setup** (as in notebook 03): task A = digits, task B = the same digits rotated 90°. Train on A, then continue
training on B only, and track A's accuracy every epoch.

In [ ]:
rotate = lambda X_: np.ascontiguousarray(np.rot90(X_.reshape(-1, 8, 8), 1, axes=(1, 2))).reshape(-1, 64)
XtrB, XteB = rotate(Xtr), rotate(Xte)

torch.manual_seed(0)
model_A = mlp(128)
opt = torch.optim.Adam(model_A.parameters(), lr=2e-3)
for epoch in range(20):
    perm = torch.randperm(len(Xt))
    for i in range(0, len(Xt), 64):
        idx = perm[i:i + 64]
        loss = F.cross_entropy(model_A(Xt[idx]), yt[idx]); opt.zero_grad(); loss.backward(); opt.step()
model_A.eval()
print(f"after training on A: acc A {accuracy(model_A, Xte, yte):.3f}, acc B {accuracy(model_A, XteB, yte):.3f}")

In [ ]:
class LoRALinear(nn.Module):                         # the module from notebook 03, compact
    def __init__(self, base, r=4, alpha=8):
        super().__init__()
        self.base = base
        self.A = nn.Parameter(torch.randn(r, base.in_features) / math.sqrt(base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        self.scale = alpha / r
    def forward(self, x):
        return self.base(x) + self.scale * (x @ self.A.T @ self.B.T)

def continue_on_B(method, epochs=15, seed=0):
    torch.manual_seed(seed)
    m = copy.deepcopy(model_A)
    lr, replay, l2 = 1e-3, 0.0, 0.0
    if method == "lower LR (1e-4)":
        lr = 1e-4
    if method == "replay 20% of A":
        replay = 0.2
    if method == "L2 to original weights":
        l2 = 0.5
    if method == "LoRA r=4 (base frozen)":
        for p in m.parameters():
            p.requires_grad_(False)
        for i, mod in enumerate(m):
            if isinstance(mod, nn.Linear):
                m[i] = LoRALinear(mod)
        lr = 3e-3
    params = [p for p in m.parameters() if p.requires_grad]
    anchor = [p.detach().clone() for p in params]
    opt_ = torch.optim.Adam(params, lr=lr)
    XB, yB = T_(XtrB), T_(ytr)
    g = torch.Generator().manual_seed(seed)
    n_rep = int(replay * len(XB))
    curve = [(accuracy(m, Xte, yte), accuracy(m, XteB, yte))]
    for epoch in range(epochs):
        m.train()
        Xe, ye = XB, yB
        if n_rep:                                    # mix a random slice of task-A data into every epoch
            ridx = torch.randperm(len(Xt), generator=g)[:n_rep]
            Xe, ye = torch.cat([XB, Xt[ridx]]), torch.cat([yB, yt[ridx]])
        perm = torch.randperm(len(Xe), generator=g)
        for i in range(0, len(Xe), 64):
            idx = perm[i:i + 64]
            loss = F.cross_entropy(m(Xe[idx]), ye[idx])
            if l2:
                loss = loss + l2 * sum(((p - a) ** 2).sum() for p, a in zip(params, anchor))
            opt_.zero_grad(); loss.backward(); opt_.step()
        m.eval()
        curve.append((accuracy(m, Xte, yte), accuracy(m, XteB, yte)))
    return np.array(curve)

methods = ["plain fine-tuning", "lower LR (1e-4)", "replay 20% of A", "L2 to original weights", "LoRA r=4 (base frozen)"]
t0 = time.time()
curves = {mth: continue_on_B(mth) for mth in methods}
print(f"{len(methods)} continual-training runs in {time.time() - t0:.1f}s")
print(f"{'method':26s} {'acc A':>6} {'acc B':>6}   (after 15 epochs on B)")
for mth, c in curves.items():
    print(f"{mth:26s} {c[-1, 0]:6.3f} {c[-1, 1]:6.3f}")

In [ ]:
colors = dict(zip(methods, ["#e03131", "#fab005", "#2f9e44", "#3b5bdb", "#ae3ec9"]))
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
for mth, c in curves.items():
    a1.plot(c[:, 0], label=mth, color=colors[mth], lw=2)
    a1.plot(c[:, 1], color=colors[mth], ls=":", lw=1)
a1.set_xlabel("epochs of training on task B"); a1.set_ylabel("test accuracy")
a1.set_title("Task A accuracy (solid) while learning B (dotted)"); a1.legend(fontsize=8)
for mth, c in curves.items():
    a2.scatter(c[-1, 1], c[-1, 0], s=80, color=colors[mth], label=mth)
a2.set_xlabel("task B accuracy (learned the new thing)"); a2.set_ylabel("task A accuracy (kept the old thing)")
a2.set_xlim(0, 1.02); a2.set_ylim(0, 1.02); a2.plot([0, 1], [1, 1], color="grey", lw=0.5)
a2.set_title("Learn vs forget: top-right is best"); plt.tight_layout(); plt.show()
assert curves["plain fine-tuning"][-1, 0] < curves["plain fine-tuning"][0, 0] - 0.2          # A collapses
assert curves["replay 20% of A"][-1, 0] > curves["plain fine-tuning"][-1, 0] + 0.2           # replay protects A

**Read the charts.** Plain fine-tuning learns B and **collapses** A within a few epochs — catastrophic forgetting.
The mitigations sit at different places on the learn-vs-forget trade-off:

- **Replay** (mixing some old-task data into training) keeps A *and* learns B — in LLM terms, mix 5–20% general
  instruction data into your domain fine-tune (`tn12`, `tn21`). It is usually the most effective single fix.
- **Lower learning rate** and **L2-to-original** (a leash on the weights, like the KL-to-reference in RLHF/DPO) do
  *not* rescue A in this run: after 15 epochs the low-LR model is half-way — already worse on A, not yet good on B —
  and the L2 run ends up mediocre at both. Slowing or leashing the update is not the same as protecting the old
  task; the path between "good at A" and "good at B" can pass through "good at neither" (the merging exercise below
  shows the same valley). Replay works because it gives the gradient an explicit reason to keep A.
- **LoRA** changes the function through a small low-rank update and lands at a useful middle point here; with the
  adapter on it still forgets (notebook 03 showed this grows with rank), but the base is untouched — switch the
  adapter off and A is back exactly.

What to do in practice: combine replay with LoRA and a sensible LR, keep epochs at 1–3, and — the step teams skip —
run a **regression eval on capabilities you did not train** (instruction-following, safety, general knowledge)
before shipping (`tn20`).

| Mitigation | How | Cost |
|---|---|---|
| replay / data mixing | 5–20% general or old-task data in every batch | needs that data; slightly more compute |
| LoRA / freezing | train a small subset; base recoverable | lower capacity for big changes |
| lower LR, fewer epochs | 1e-5–5e-5 full FT, 1e-4–3e-4 LoRA; 1–3 epochs | may under-fit the new task |
| L2 / KL to the original | penalise distance from the base weights or outputs (EWC weights it by importance) | a hyperparameter to tune |
| model merging | average the fine-tune with the base (or with other fine-tunes) | empirical; measure (`tn27`) |

## Try it yourself

**1.** Sweep the replay fraction (0%, 5%, 10%, 20%, 50%) and plot final A and B accuracy. How little old data is
enough here?

In [ ]:
# Solution — try it yourself first, then run this
fractions = [0.0, 0.05, 0.1, 0.2, 0.5]
def replay_run(frac, epochs=15):
    torch.manual_seed(0)
    m = copy.deepcopy(model_A); opt_ = torch.optim.Adam(m.parameters(), lr=1e-3)
    g = torch.Generator().manual_seed(0)
    XB, yB = T_(XtrB), T_(ytr)
    for _ in range(epochs):
        ridx = torch.randperm(len(Xt), generator=g)[:int(frac * len(XB))]
        Xe, ye = torch.cat([XB, Xt[ridx]]), torch.cat([yB, yt[ridx]])
        perm = torch.randperm(len(Xe), generator=g)
        for i in range(0, len(Xe), 64):
            idx = perm[i:i + 64]; loss = F.cross_entropy(m(Xe[idx]), ye[idx]); opt_.zero_grad(); loss.backward(); opt_.step()
    return accuracy(m, Xte, yte), accuracy(m, XteB, yte)
res = [replay_run(f) for f in fractions]
for f, (a, b) in zip(fractions, res):
    print(f"replay {f:4.0%}: acc A {a:.3f}  acc B {b:.3f}")
plt.figure(figsize=(6, 3.2))
plt.plot([f * 100 for f in fractions], [a for a, _ in res], "o-", label="task A (old)")
plt.plot([f * 100 for f in fractions], [b for _, b in res], "s-", label="task B (new)")
plt.xlabel("% of old-task data mixed in"); plt.ylabel("test accuracy"); plt.legend(); plt.title("A little replay goes a long way"); plt.show()

**2.** Model merging in miniature: average the weights of `model_A` and the plainly fine-tuned model
(`θ = (1 − λ)·θ_A + λ·θ_B`) for λ from 0 to 1. Is there a λ that is decent at both tasks?

In [ ]:
# Solution — try it yourself first, then run this
torch.manual_seed(0)
model_B = copy.deepcopy(model_A); opt_ = torch.optim.Adam(model_B.parameters(), lr=1e-3)
XB, yB = T_(XtrB), T_(ytr)
for _ in range(15):
    perm = torch.randperm(len(XB))
    for i in range(0, len(XB), 64):
        idx = perm[i:i + 64]; loss = F.cross_entropy(model_B(XB[idx]), yB[idx]); opt_.zero_grad(); loss.backward(); opt_.step()
for lam in (0.0, 0.25, 0.5, 0.75, 1.0):
    merged = copy.deepcopy(model_A)
    with torch.no_grad():
        for pm, pa, pb in zip(merged.parameters(), model_A.parameters(), model_B.parameters()):
            pm.copy_((1 - lam) * pa + lam * pb)
    print(f"λ={lam:.2f}: acc A {accuracy(merged, Xte, yte):.3f}  acc B {accuracy(merged, XteB, yte):.3f}")
print("Merging works when both models started from the same weights; whether a good middle exists must be measured.")

**3.** Distillation temperature sweep: train the transfer-set student with T = 1, 2, 4, 8. Does temperature matter
when the transfer set is large?

In [ ]:
# Solution — try it yourself first, then run this
all_idx = np.concatenate([lab_idx, unl_idx])
for T in (1.0, 2.0, 4.0, 8.0):
    s = train_student(all_idx, use_soft=True, T=T, hard_idx=lab_idx)
    print(f"T={T:.0f}: student accuracy {accuracy(s, Xte, yte):.3f}")
print("With a large teacher-labelled transfer set T moves accuracy by only a few points (here low T is best): the")
print("transfer set itself carries most of the signal. High T matters most when the student sees few examples.")

## Say it in an interview

- **Distillation in 30 seconds:** "Train a small student to reproduce a big teacher. With LLMs it's usually response
  distillation — the teacher writes answers to real prompts, you filter them and SFT a small model. Classic logit
  distillation matches the teacher's softened distribution: α·T²·KL(teacher/T ‖ student/T) + (1−α)·CE; T spreads the
  probabilities so the student sees which classes are similar, T² keeps the gradient scale constant."
- **It's a cost project:** the student approaches the teacher on the distilled task at a fraction of the size and
  latency; it cannot exceed the teacher there and inherits its errors. Check the teacher's terms of service.
- **Forgetting in 30 seconds:** "Fine-tuning moves weights the old capabilities depended on. Mitigate with replay
  (5–20% general data), LoRA or freezing, a low LR and 1–3 epochs, and a KL/L2 leash to the base — then prove it with
  a regression eval on what you did not train, including safety."
- **Traps:** "LoRA can't forget" (it can with the adapter on — but the base is recoverable); reporting only the
  task metric ("+12 points") with no general-capability regression numbers; training on synthetic data from the same
  model family you evaluate with.

## Next

- [08 · OpenAI fine-tuning API](08_openai_fine_tuning_api.ipynb) — hosted fine-tuning (a common way to train the student)
- [03 · LoRA from scratch](03_lora_from_scratch.ipynb) · [06 · DPO preference tuning](06_dpo_preference_tuning.ipynb)
- [Overfitting, regularization and cross-validation](../05_machine_learning/05_overfitting_regularization_cross_validation.ipynb)
- [Eval harness from scratch](../17_interview_coding_rounds/05_eval_harness_from_scratch.ipynb) — build the regression suite
- Theory: [genai_flow course](../../genai_flow/index.html) · [interview bank](../../ai_interview_prep/index.html)